
## 🧰 Core Concepts Covered

### 1. **PydanticOutputParser** 📋
Parse LLM outputs into type-safe Pydantic models with automatic validation

### 2. **RunnableLambda** ⛓️
Create custom LangChain chains using Python lambda functions

### 3. **PromptTemplate.partial()** 🧩
Pre-fill template variables (like format instructions) for cleaner, reusable prompts

### 4. **AgentExecutor** 🤖
Orchestrate the agent's reasoning loop: Think → Act → Observe → Repeat

---

## 1. Imports and Environment Setup

In [19]:
from langchain_classic import hub
from langchain_classic.agents import AgentExecutor
from langchain_classic.agents.react.agent import create_react_agent
from langchain_tavily import TavilySearch
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnableLambda
from prompts import main_prompt
from schema import AgentResponse
from langchain_core.output_parsers.pydantic import PydanticOutputParser
from dotenv import load_dotenv

# Load environment variables
load_dotenv()

True

## 2. Initialize Tools

In [20]:
# Initialize Tavily search tool
tavily_search = TavilySearch(max_results=5, topic="general")
tools = [tavily_search]

print(f"Tools initialized: {[tool.name for tool in tools]}")

Tools initialized: ['tavily_search']


## 3. Initialize LLM

In [21]:
# Initialize Google Gemini LLM
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0
)

print("LLM initialized successfully")

LLM initialized successfully


## 4. 📋 Setup Output Parser and Prompt

### The Problem
LLMs return unstructured text. We need **structured, validated data**.

### The Solution: PydanticOutputParser
Converts LLM JSON responses into type-safe Pydantic objects with automatic validation.

**How it works:**
1. Define your data structure using a Pydantic model
2. Parser generates format instructions for the LLM
3. LLM returns JSON matching your schema
4. Parser validates and converts to Python objects

### The `.partial()` Method
Pre-fills the `format_instructions` variable in our prompt template, so we don't have to pass it every time we invoke the agent.

In [22]:
# Initialize Pydantic output parser
output_parser = PydanticOutputParser(pydantic_object=AgentResponse)

# Get format instructions
format_instructions = output_parser.get_format_instructions()
print("Format Instructions:")
print(format_instructions)

Format Instructions:
The output should be formatted as a JSON instance that conforms to the JSON schema below.

As an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}
the object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.

Here is the output schema:
```
{"$defs": {"Source": {"properties": {"url": {"description": "URL of the source", "title": "Url", "type": "string"}}, "required": ["url"], "title": "Source", "type": "object"}}, "properties": {"answer": {"description": "agent's answer to the question", "title": "Answer", "type": "string"}, "sources": {"description": "List of sources used to answer the question", "items": {"$ref": "#/$defs/Source"}, "title": "Sources", "type": "array"}}, "required": ["answer"]}
```


In [23]:
# Create prompt template with partial format instructions
react_prompt_with_format_instructions = PromptTemplate(
    template=main_prompt, 
    input_variables=["input", "agent_scratchpad", "tool_names"]
).partial(format_instructions=output_parser.get_format_instructions())

print("Prompt template created with format instructions")

Prompt template created with format instructions


## 5. Create Agent and Executor

In [24]:
# Create ReAct agent
agent = create_react_agent(
    llm=llm, 
    tools=tools, 
    prompt=react_prompt_with_format_instructions
)

print("Agent created successfully")

Agent created successfully


### 💡 Understanding `{tools}` and `{tool_names}`

You might notice that our prompt template uses `{tools}` and `{tool_names}`, but we didn't explicitly pass them in the `input_variables` list or the `.partial()` method. **How does this work?**

The magic happens inside `create_react_agent`:
1. It takes the list of `tools` you provide.
2. It automatically generates a text description for each tool and populates the `{tools}` variable.
3. It creates a comma-separated list of tool names and populates the `{tool_names}` variable.

This is why you don't need to manually handle these variables—LangChain does it for you!

In [25]:
# Create agent executor
agent_executor = AgentExecutor(
    agent=agent, 
    tools=tools, 
    verbose=True, 
    handle_parsing_errors=True
)

chain = agent_executor
print("Agent executor created successfully")

Agent executor created successfully


## 6. Run the Agent

in the below response you will see a line <br>
```Invalid Format: Missing 'Action:' after 'Thought:'``` <br>
this is because the response is not in the correct format
and so the whole flow re itertes 

In [26]:
# Execute the agent with a query
result = chain.invoke(input={"input": "What are the latest news about AI in the world"})
print("\n" + "="*50)
print("FINAL RESULT:")
print("="*50)
print(result)



> Entering new AgentExecutor chain...
Action: tavily_search
Action Input: latest global AI news{'query': 'latest global AI news', 'follow_up_questions': None, 'answer': None, 'images': [], 'results': [{'url': 'https://www.artificialintelligence-news.com/', 'title': 'AI News | Latest News | Insights Powering AI-Driven Business Growth', 'content': 'AI News delivers the latest updates in artificial intelligence, machine learning, deep learning, enterprise AI, and emerging tech worldwide.', 'score': 0.74541736, 'raw_content': None}, {'url': 'https://www.reuters.com/technology/artificial-intelligence/', 'title': 'AI News | Latest Headlines and Developments | Reuters', 'content': 'Artificial Intelligence · 8 hours ago. Micron to invest $9.6 billion in Japan to build AI memory chip plant, Nikkei reports · 7 hours ago. Ukrainian drone pilots', 'score': 0.6347929, 'raw_content': None}, {'url': 'https://www.wsj.com/tech/ai?gaa_at=eafs&gaa_n=AWEtsqdjtYQJ9GMxKJEsknTUY2ZRuo2ELKsIZ1g-WTX5Ogvz4YFPR

## 7. Debug Section (Optional)
Use the cells below to inspect individual components

In [27]:
# Inspect the full prompt template
print("Full Prompt Template:")
print(main_prompt)

Full Prompt Template:

Answer the following questions as best you can. You have access to the following tools:

{tools}

Use the following format:

Question: the input question you must answer
Thought: you should always think about what to do
Action: the action to take, should be one of [{tool_names}]
Action Input: the input to the action
Observation: the result of the action
... (this Thought/Action/Action Input/Observation can repeat N times)
Thought: I now know the final answer
Final Answer: the final answer to the original input question formatted according to the schema:

{format_instructions}

Begin!

Question: {input}
Thought:{agent_scratchpad}



In [28]:
# Inspect tools
for tool in tools:
    print(f"Tool Name: {tool.name}")
    print(f"Tool Description: {tool.description}")
    print("-" * 50)

Tool Name: tavily_search
Tool Description: A search engine optimized for comprehensive, accurate, and trusted results. Useful for when you need to answer questions about current events. It not only retrieves URLs and snippets, but offers advanced search depths, domain management, time range filters, and image search, this tool delivers real-time, accurate, and citation-backed results.Input should be a search query.
--------------------------------------------------


### the data type if result from the agent is a DICTIONARY
and it has 2 keys ['input' , 'output']
and also the output is in the form of a string which <br> is actually a json string
so we need to convert it into a json object

In [33]:
print(type(result))
print(result.keys())
print(result['output'])

<class 'dict'>
dict_keys(['input', 'output'])
```json
{
 "answer": "Recent global AI news highlights several key developments:\n\n*   **Investment Growth:** Generative AI continues to attract significant private investment, reaching $33.9 billion globally, which represents an 18.7% increase from 2023.\n*   **Hardware Manufacturing:** Micron plans to invest $9.6 billion in Japan to construct a new plant dedicated to producing AI memory chips, signaling a major expansion in AI hardware infrastructure.\n*   **Workforce Integration:** The adoption of AI among workers is currently observed to be slow and uneven across industries.\n*   **Regulatory Landscape:** Major tech companies are reportedly accumulating substantial funds to influence and potentially combat AI regulation efforts. Concurrently, regulatory bodies, such as Italy's competition watchdog, are broadening their scrutiny of AI-related practices.\n*   **Military Applications:** There are ongoing reports regarding the use of AI in

below you can see that the output parser <br>
actualy returns an object of type AgentResponse

In [35]:
parsed_result= output_parser.parse(result["output"])
print(parsed_result)
print(type(parsed_result))

answer="Recent global AI news highlights several key developments:\n\n*   **Investment Growth:** Generative AI continues to attract significant private investment, reaching $33.9 billion globally, which represents an 18.7% increase from 2023.\n*   **Hardware Manufacturing:** Micron plans to invest $9.6 billion in Japan to construct a new plant dedicated to producing AI memory chips, signaling a major expansion in AI hardware infrastructure.\n*   **Workforce Integration:** The adoption of AI among workers is currently observed to be slow and uneven across industries.\n*   **Regulatory Landscape:** Major tech companies are reportedly accumulating substantial funds to influence and potentially combat AI regulation efforts. Concurrently, regulatory bodies, such as Italy's competition watchdog, are broadening their scrutiny of AI-related practices.\n*   **Military Applications:** There are ongoing reports regarding the use of AI in military contexts, including by Ukrainian drone pilots.\n\n

### Ideally this should be the approach
1. you create the create_react_agent
2. you create the agent_executor
3. you create the fetch_output (RunnableLambda)
4. you create the parse_json_output (RunnableLambda)
5. you create the chain
like what is written below

In [36]:
agent= create_react_agent(llm=llm, tools=tools, prompt=react_prompt_with_format_instructions)
agent_executor= AgentExecutor(agent=agent, tools=tools, verbose=True, handle_parsing_errors=True)
fetch_output= RunnableLambda(lambda x: x["output"])
parse_json_output= RunnableLambda(lambda x:output_parser.parse(x))


chain= agent_executor | fetch_output | parse_json_output

# now when you invoke this chain you get the final answer
# this is the more langchain way of doing it



so this is majorly all about the react core agent
this is how it works

